# Review 3. UMAP hyperparameter survey

The UMAP step of notebook 2 repeated over a grid of settings.  HDBSCAN is not run
here: the survey is about the embedding, and the review figures are drawn from
the coordinates alone.

| input | what it provides |
| --- | --- |
| `metadata/params.csv` | one row per spectrum: the table built in notebook 0; fixes the row order |
| `data/review/F<S>_norm.csv` | the continuum removed fluxes of review notebook 1, at S = 10, 20, 30, 40, 50 A |

Three things are varied against the notebook 2 baseline (30 A, 5100-6450 A,
`n_neighbors = 12`, `random_state = 50`, `min_dist = 0`, `metric = 'euclidean'`):

1. **Smoothing scale** -- the five tables on the 5100-6450 A window.
2. **Wavelength window** -- the 30 A table with the window widened by 50 or 100 A
   on the blue side, the red side, or both.
3. **`min_dist` and `metric`** -- the baseline table and window.

For 1 and 2 every pair of `n_neighbors` in 2..30 and `random_state` in 40..58 is
embedded: 290 embeddings per table, ten tables.  For 3 the two are held at the
baseline.

| output | columns appended to `params.csv` |
| --- | --- |
| `results/review/survey_neighbors_<tag>_<window>_dim2.csv` | `umap_x_<n_neighbors>_r<random_state>`, `umap_y_...` |
| `results/review/survey_mindist_metric_F30_5100_6450_dim2_nei12_r50.csv` | `umap_x_mindist_<min_dist>_<metric>`, `umap_y_...` |


## Imports

In [1]:
import os
import pandas as pd

In [2]:
import umap

In [3]:
import matplotlib.pyplot as plt

## Configuration


In [4]:
# Input / output locations.  The spectrum tables are the ones review notebook 1
# builds, keyed by the smoothing scale tag it names them with.
PARAMS_PATH = './metadata/params.csv'
SPECTRUM_TABLE_DIR = './data/review'
SPECTRUM_TABLE_PATHS = {tag: f'{SPECTRUM_TABLE_DIR}/{tag}_norm.csv'
                        for tag in ['F20', 'F30', 'F40']}

RESULTS_DIR = './results/review'

# Column label format written by notebook 1: '<SN name>%%%<SN type>$$$<phase>'
NAME_TYPE_SEPARATOR = '%%%'

In [5]:
# Baseline: the notebook 2 settings.  Every survey varies some of these and holds
# the rest fixed.
BASE_SPECTRUM_TAG = 'F30'
BASE_WINDOW_AA = (5100, 6450)

UMAP_N_COMPONENTS = 2
UMAP_N_NEIGHBORS = 12
UMAP_MIN_DIST = 0.0
UMAP_METRIC = 'euclidean'
UMAP_RANDOM_STATE = 50

In [6]:
# Survey 1 and 2: the (spectrum table, wavelength window) settings on which every
# (n_neighbors, random_state) pair is embedded.  The baseline is the first entry.
SURVEY_N_NEIGHBORS = range(2, 31)
SURVEY_RANDOM_STATES = range(40, 60, 2)

SMOOTHING_SURVEY = [
    ('F30', (5100, 6450)),   # baseline
    ('F20', (5100, 6450)),
    ('F40', (5100, 6450)),
]

WINDOW_SURVEY = [
    ('F30', (5050, 6450)),   # blue edge  -50 A
    ('F30', (5000, 6450)),   # blue edge -100 A
    ('F30', (5100, 6500)),   # red edge   +50 A
    ('F30', (5100, 6550)),   # red edge  +100 A
    ('F30', (5050, 6500)),   # both edges +-50 A
]

NEIGHBOR_SURVEY_SETTINGS = SMOOTHING_SURVEY + WINDOW_SURVEY

In [7]:
# Survey 3: every (min_dist, metric) pair on the baseline table and window, with
# n_neighbors and random_state held at the baseline.
SURVEY_MIN_DISTS = [0.0, 0.05, 0.1]
SURVEY_METRICS = ['euclidean', 'manhattan', 'cosine']

In [8]:
# The settings name the outputs.
def neighbor_survey_path(spectrum_tag, window):
    """Output table of survey 1 / 2 for one (spectrum table, window) setting."""
    window_min, window_max = window
    return (f'{RESULTS_DIR}/survey_neighbors_{spectrum_tag}'
            f'_{window_min}_{window_max}_dim{UMAP_N_COMPONENTS}.csv')


MINDIST_METRIC_SURVEY_PATH = (
    f'{RESULTS_DIR}/survey_mindist_metric_{BASE_SPECTRUM_TAG}'
    f'_{BASE_WINDOW_AA[0]}_{BASE_WINDOW_AA[1]}_dim{UMAP_N_COMPONENTS}'
    f'_nei{UMAP_N_NEIGHBORS}_r{UMAP_RANDOM_STATE}.csv'
)

for spectrum_tag, window in NEIGHBOR_SURVEY_SETTINGS:
    print(neighbor_survey_path(spectrum_tag, window))
print(MINDIST_METRIC_SURVEY_PATH)

./results/review/survey_neighbors_F30_5100_6450_dim2.csv
./results/review/survey_neighbors_F20_5100_6450_dim2.csv
./results/review/survey_neighbors_F40_5100_6450_dim2.csv
./results/review/survey_neighbors_F30_5050_6450_dim2.csv
./results/review/survey_neighbors_F30_5000_6450_dim2.csv
./results/review/survey_neighbors_F30_5100_6500_dim2.csv
./results/review/survey_neighbors_F30_5100_6550_dim2.csv
./results/review/survey_neighbors_F30_5050_6500_dim2.csv
./results/review/survey_mindist_metric_F30_5100_6450_dim2_nei12_r50.csv


## Reading the inputs


In [9]:
def load_params(params_path=PARAMS_PATH):
    """Read the metadata table built in notebook 0.

    Its row order becomes the row order of every results table.
    """
    return pd.read_csv(params_path)

In [10]:
def load_fluxes(params, spectrum_tag=BASE_SPECTRUM_TAG, window=BASE_WINDOW_AA):
    """Read one normalized spectrum table of notebook 1 as one row per supernova.

    The table holds one column per spectrum, labelled
    `<SN name>%%%<SN type>$$$<phase>`, and one row per wavelength bin.  It is cut
    to the window given, transposed so that each supernova becomes a row and each
    wavelength a feature, and reordered to follow the rows of `params` so that the
    embedding can be written straight back next to the metadata.
    """
    window_min, window_max = window

    spectra = pd.read_csv(SPECTRUM_TABLE_PATHS[spectrum_tag]).set_index('AA')
    spectra = spectra[(spectra.index > window_min) & (spectra.index < window_max)]

    column_of = {column.split(NAME_TYPE_SEPARATOR)[0]: column for column in spectra.columns}
    if len(column_of) != len(spectra.columns):
        raise ValueError('the spectrum table holds more than one spectrum per supernova')

    # A missing supernova raises a KeyError naming it
    ordered = spectra[[column_of[name] for name in params['SN_name']]]

    return ordered.T.reset_index(drop=True)

## Embedding

`embed` is the notebook 2 function.  The two survey functions call it once per
grid point and collect the coordinates into one table, two columns per setting.


In [11]:
def embed(fluxes, n_components=UMAP_N_COMPONENTS, n_neighbors=UMAP_N_NEIGHBORS,
          min_dist=UMAP_MIN_DIST, metric=UMAP_METRIC, random_state=UMAP_RANDOM_STATE):
    """UMAP embedding of the fluxes, one row per supernova."""
    reducer = umap.UMAP(n_components=n_components, n_neighbors=n_neighbors,
                        min_dist=min_dist, metric=metric, random_state=random_state)

    return reducer.fit_transform(fluxes)

In [12]:
def survey_neighbors(fluxes, n_neighbors_grid=SURVEY_N_NEIGHBORS,
                     random_state_grid=SURVEY_RANDOM_STATES):
    """Embeddings for every (n_neighbors, random_state) pair, as one table.

    Columns are `umap_x_<n_neighbors>_r<random_state>` and the `umap_y` partner.
    """
    columns = {}
    for n_neighbors in n_neighbors_grid:
        for random_state in random_state_grid:
            embedding = embed(fluxes, n_neighbors=n_neighbors, random_state=random_state)
            columns[f'umap_x_{n_neighbors}_r{random_state}'] = embedding[:, 0]
            columns[f'umap_y_{n_neighbors}_r{random_state}'] = embedding[:, 1]

    return pd.DataFrame(columns)

In [13]:
def survey_mindist_metric(fluxes, min_dist_grid=SURVEY_MIN_DISTS, metric_grid=SURVEY_METRICS):
    """Embeddings for every (min_dist, metric) pair, as one table.

    Columns are `umap_x_mindist_<min_dist>_<metric>` and the `umap_y` partner.
    """
    columns = {}
    for min_dist in min_dist_grid:
        for metric in metric_grid:
            embedding = embed(fluxes, min_dist=min_dist, metric=metric)
            columns[f'umap_x_mindist_{min_dist}_{metric}'] = embedding[:, 0]
            columns[f'umap_y_mindist_{min_dist}_{metric}'] = embedding[:, 1]

    return pd.DataFrame(columns)

## Run

The neighbour / seed survey is 290 embeddings per setting and ten settings; it
takes some minutes.  Each table is kept in `survey_tables`, keyed by its output
path, until the write section below.


In [14]:
params = load_params()
survey_tables = {}

In [15]:
for spectrum_tag, window in NEIGHBOR_SURVEY_SETTINGS:
    fluxes = load_fluxes(params, spectrum_tag, window)
    print(f'{spectrum_tag} {window[0]}-{window[1]} A: '
          f'{len(fluxes)} spectra x {fluxes.shape[1]} wavelength bins ...', end=' ')

    coordinates = survey_neighbors(fluxes)
    survey_tables[neighbor_survey_path(spectrum_tag, window)] = pd.concat([params, coordinates], axis=1)
    print(f'{coordinates.shape[1] // 2} embeddings')

F30 5100-6450 A: 119 spectra x 675 wavelength bins ... 290 embeddings
F20 5100-6450 A: 119 spectra x 675 wavelength bins ... 290 embeddings
F40 5100-6450 A: 119 spectra x 675 wavelength bins ... 290 embeddings
F30 5050-6450 A: 119 spectra x 700 wavelength bins ... 290 embeddings
F30 5000-6450 A: 119 spectra x 724 wavelength bins ... 290 embeddings
F30 5100-6500 A: 119 spectra x 700 wavelength bins ... 290 embeddings
F30 5100-6550 A: 119 spectra x 725 wavelength bins ... 290 embeddings
F30 5050-6500 A: 119 spectra x 725 wavelength bins ... 290 embeddings


In [16]:
fluxes = load_fluxes(params, BASE_SPECTRUM_TAG, BASE_WINDOW_AA)
coordinates = survey_mindist_metric(fluxes)
survey_tables[MINDIST_METRIC_SURVEY_PATH] = pd.concat([params, coordinates], axis=1)
print(f'{BASE_SPECTRUM_TAG} {BASE_WINDOW_AA[0]}-{BASE_WINDOW_AA[1]} A: '
      f'{coordinates.shape[1] // 2} embeddings')

survey_tables[MINDIST_METRIC_SURVEY_PATH].head()

F30 5100-6450 A: 9 embeddings


,SN_name,RA,DEC,redshift,host_name,Host_RA,Host_DEC,logd25,e_logd25,logr25,...,umap_x_mindist_0.05_manhattan,umap_y_mindist_0.05_manhattan,umap_x_mindist_0.05_cosine,umap_y_mindist_0.05_cosine,umap_x_mindist_0.1_euclidean,umap_y_mindist_0.1_euclidean,umap_x_mindist_0.1_manhattan,umap_y_mindist_0.1_manhattan,umap_x_mindist_0.1_cosine,umap_y_mindist_0.1_cosine
0,SN1994D,188.51021,7.70131,0.001494,NGC4526,188.512545,7.699261,1.842,0.018,0.449,...,13.067949,5.983602,10.690466,11.549214,7.922316,8.565357,5.493999,7.492243,5.094380,7.569849
1,SN1994S,187.84108,29.13450,0.015177,NGC4495,187.845363,29.136442,1.128,0.035,0.373,...,12.465940,5.474339,9.959555,11.938252,8.791357,7.806912,5.781505,7.100060,4.146730,7.548430
2,SN1996ai,197.74221,37.05983,0.002900,NGC5005,197.734470,37.058994,1.683,0.025,0.500,...,11.426907,4.385129,8.797357,10.291723,7.687937,6.482667,4.522971,5.565524,3.858945,5.567180
3,SN1996C,207.70250,49.31864,0.027000,PGC049153,207.703404,49.315106,0.899,0.064,0.257,...,12.313672,4.346304,9.879884,10.931593,7.760892,7.879569,4.673840,6.570879,4.553167,6.649789
4,SN1996X,199.50471,-26.84592,0.008876,NGC5061,199.521255,-26.837131,1.574,0.027,0.092,...,14.109294,4.900505,10.902008,9.038623,5.964146,8.905493,7.570417,6.161934,6.562678,5.615366


## Write the tables


In [18]:
os.makedirs(RESULTS_DIR, exist_ok=True)

for path, table in survey_tables.items():
    table.to_csv(path, index=False)
    print(f'written to {path}')

written to ./results/review/survey_neighbors_F30_5100_6450_dim2.csv
written to ./results/review/survey_neighbors_F20_5100_6450_dim2.csv
written to ./results/review/survey_neighbors_F40_5100_6450_dim2.csv
written to ./results/review/survey_neighbors_F30_5050_6450_dim2.csv
written to ./results/review/survey_neighbors_F30_5000_6450_dim2.csv
written to ./results/review/survey_neighbors_F30_5100_6500_dim2.csv
written to ./results/review/survey_neighbors_F30_5100_6550_dim2.csv
written to ./results/review/survey_neighbors_F30_5050_6500_dim2.csv
written to ./results/review/survey_mindist_metric_F30_5100_6450_dim2_nei12_r50.csv
